# Step 3: SQL Analysis
Data: startup_funding_clean.csv (5,857 deals, 2015 – May 2022), created in notebook 01_merge_and_clean.

In [ ]:
import pandas as pd, sqlite3

clean = pd.read_csv('startup_funding_clean.csv')
conn = sqlite3.connect('startups.db')                 # creates a database file
clean.to_sql('funding', conn, if_exists='replace', index=False)   # table named 'funding'

def q(sql):                                           # small helper to run a query
    return pd.read_sql(sql, conn)

q("SELECT COUNT(*) AS total_rows FROM funding")

,total_rows
0,5857


In [ ]:
q("""
SELECT Year,
       COUNT(*)                          AS deals,
       ROUND(SUM(Amount_USD)/1e9, 2)     AS total_funding_bn,
       ROUND(AVG(Amount_USD)/1e6, 2)     AS avg_deal_mn
FROM funding
GROUP BY Year
ORDER BY Year
""")

,Year,deals,total_funding_bn,avg_deal_mn
0,2015,934,8.67,13.30
1,2016,992,3.83,6.54
2,2017,687,10.43,22.87
3,2018,525,6.61,17.55
4,2019,89,3.34,43.33
5,2020,1048,17.84,22.33
6,2021,1187,29.27,28.25
7,2022,395,16.71,47.74


In [ ]:
q("""
SELECT Sector,
       COUNT(*)                          AS deals,
       ROUND(SUM(Amount_USD)/1e9, 2)     AS total_funding_bn
FROM funding
WHERE Sector NOT IN ('Unknown', 'Other')
GROUP BY Sector
ORDER BY total_funding_bn DESC
LIMIT 10
""")

,Sector,deals,total_funding_bn
0,E-commerce & Retail,716,18.56
1,Tech & SaaS,1134,15.42
2,FinTech,604,14.50
3,HealthTech,449,7.45
4,Logistics & Mobility,309,7.19
5,Food & Beverage,260,6.21
6,Media & Entertainment,231,4.68
7,Consumer Internet,848,4.51
8,EdTech,166,2.51
9,Business Services,202,2.08


In [ ]:
q("""
SELECT CASE WHEN City_Std IN ('Delhi','Gurugram','Noida') THEN 'Delhi-NCR'
            ELSE City_Std END            AS city_group,
       COUNT(*)                          AS deals,
       ROUND(SUM(Amount_USD)/1e9, 2)     AS total_funding_bn
FROM funding
WHERE City_Std NOT IN ('Unknown', 'Outside India')
GROUP BY city_group
ORDER BY total_funding_bn DESC
LIMIT 10
""")

,city_group,deals,total_funding_bn
0,Bengaluru,1761,41.75
1,Delhi-NCR,1583,20.62
2,Mumbai,1027,16.73
3,Chennai,212,2.81
4,Pune,205,2.47
5,Haryana,30,1.01
6,Jaipur,63,0.83
7,Hyderabad,178,0.82
8,Faridabad,6,0.43
9,Shanghai,1,0.40


In [ ]:
q("""
SELECT Stage_Std,
       COUNT(*)                          AS deals,
       ROUND(AVG(Amount_USD)/1e6, 2)     AS avg_deal_mn,
       ROUND(SUM(Amount_USD)/1e9, 2)     AS total_funding_bn
FROM funding
WHERE Stage_Std <> 'Unknown/Other'
GROUP BY Stage_Std
ORDER BY total_funding_bn DESC
""")

,Stage_Std,deals,avg_deal_mn,total_funding_bn
0,Private Equity,1214,26.15,24.66
1,Series D+,124,138.23,16.86
2,Series C,131,49.79,6.37
3,Series B,161,29.96,4.67
4,Series A,363,12.66,4.39
5,Seed/Angel,2196,1.36,1.81
6,Pre-Series A/B,350,2.98,0.87
7,Debt,71,12.00,0.83
8,Pre-Seed,83,0.65,0.04


In [ ]:
q("""
SELECT Startup,
       COUNT(*)                          AS rounds,
       ROUND(SUM(Amount_USD)/1e6, 1)     AS total_funding_mn
FROM funding
WHERE Amount_USD IS NOT NULL
GROUP BY LOWER(Startup)
ORDER BY total_funding_mn DESC
LIMIT 10
""")

,Startup,rounds,total_funding_mn
0,Flipkart,6,4121.7
1,Swiggy,12,2826.5
2,Reliance,1,2200.0
3,Paytm,3,2140.0
4,Byju's,7,2122.0
5,Ola,7,1845.5
6,Oyo,7,1765.0
7,VerSe Innovation,3,1550.0
8,Zomato,9,1324.0
9,Dream Sports,2,1240.0


In [ ]:
q("""
SELECT Investors,
       COUNT(*) AS deals
FROM funding
WHERE Investors IS NOT NULL
  AND Investors NOT LIKE '%ndisclosed%'
GROUP BY Investors
ORDER BY deals DESC
LIMIT 10
""")

,Investors,deals
0,Inflection Point Ventures,41
1,Venture Catalysts,40
2,Indian Angel Network,29
3,Ratan Tata,25
4,Kalaari Capital,24
5,Tiger Global,23
6,Sequoia Capital,19
7,Mumbai Angels Network,18
8,Unicorn India Ventures,15
9,Trifecta Capital,15


In [ ]:
q("""
WITH yearly AS (
    SELECT Year, Sector, SUM(Amount_USD) AS total,
           RANK() OVER (PARTITION BY Year ORDER BY SUM(Amount_USD) DESC) AS rnk
    FROM funding
    WHERE Sector NOT IN ('Unknown', 'Other')
    GROUP BY Year, Sector
)
SELECT Year, Sector AS top_sector, ROUND(total/1e9, 2) AS funding_bn
FROM yearly
WHERE rnk = 1
ORDER BY Year
""")

,Year,top_sector,funding_bn
0,2015,E-commerce & Retail,3.33
1,2016,Consumer Internet,1.92
2,2017,E-commerce & Retail,5.94
3,2018,FinTech,1.56
4,2019,Tech & SaaS,1.11
5,2020,Tech & SaaS,4.71
6,2021,FinTech,6.12
7,2022,Tech & SaaS,4.22
